# Brique 1 — Modèle écologique pur

Exploration interactive de la dynamique Lotka-Volterra du framework `bilevel-fishery`.

**Objectifs**:
1. Voir les trajectoires sans pression de pêche
2. Voir l'effet d'une pêche constante
3. Comparer Euler vs RK45 pour différents `dt`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from bilevel_fishery.ecology import EcologicalState, EcologyParams, step

plt.rcParams["figure.figsize"] = (10, 5)

## 1. Trajectoire libre (pas de pêche)

Aux paramètres par défaut, l'équilibre est en
$(F^*, A^*) = (\alpha/\beta,\ \gamma/\delta) = (10,\ 20)$.
On démarre légèrement décalé pour voir les oscillations.

In [ ]:
params = EcologyParams(dt=0.05, integrator="rk45")

state = EcologicalState(fish=15.0, algae=15.0)
fish_traj = [state.fish]
algae_traj = [state.algae]

for _ in range(400):
    state = step(state, params, harvest=0.0)
    fish_traj.append(state.fish)
    algae_traj.append(state.algae)

t = np.arange(len(fish_traj)) * params.dt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(t, fish_traj, label="fish (predator)")
ax1.plot(t, algae_traj, label="algae (prey)")
ax1.axhline(
    params.alpha / params.beta, color="C0", ls=":", label=r"$F^* = \alpha/\beta$"
)
ax1.axhline(
    params.gamma / params.delta, color="C1", ls=":", label=r"$A^* = \gamma/\delta$"
)
ax1.set_xlabel("time")
ax1.set_ylabel("biomass")
ax1.set_title("Time series (no harvest)")
ax1.legend()

ax2.plot(algae_traj, fish_traj, lw=0.8)
ax2.plot(
    [params.gamma / params.delta],
    [params.alpha / params.beta],
    "r*",
    ms=12,
    label="equilibrium",
)
ax2.set_xlabel("algae")
ax2.set_ylabel("fish")
ax2.set_title("Phase portrait (closed orbit)")
ax2.legend()
plt.tight_layout()
plt.show()

## 2. Effet d'une pêche constante

On rajoute un `harvest` constant. Si la pression de pêche est trop forte,
le stock de poissons s'effondre.

In [ ]:
params = EcologyParams(dt=0.05, integrator="rk45")
harvests = [0.0, 0.5, 1.5]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, harvest in zip(axes, harvests, strict=True):
    state = EcologicalState(fish=15.0, algae=15.0)
    fish_traj = [state.fish]
    algae_traj = [state.algae]
    for _ in range(400):
        state = step(state, params, harvest=harvest)
        fish_traj.append(state.fish)
        algae_traj.append(state.algae)
    t = np.arange(len(fish_traj)) * params.dt
    ax.plot(t, fish_traj, label="fish")
    ax.plot(t, algae_traj, label="algae")
    ax.set_xlabel("time")
    ax.set_title(f"harvest = {harvest}")
    ax.legend()
axes[0].set_ylabel("biomass")
plt.tight_layout()
plt.show()

## 3. Euler vs RK45 : pourquoi le choix du solveur compte

Pour un `dt` modéré, Euler accumule de l'erreur d'amplitude
(les oscillations grandissent artificiellement).
RK45 reste fidèle à la dynamique réelle.

In [ ]:
dts = [0.05, 0.2]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, dt in zip(axes, dts, strict=True):
    p_euler = EcologyParams(dt=dt, integrator="euler")
    p_rk45 = EcologyParams(dt=dt, integrator="rk45")
    state_e = EcologicalState(fish=15.0, algae=15.0)
    state_r = EcologicalState(fish=15.0, algae=15.0)
    fish_e = [state_e.fish]
    fish_r = [state_r.fish]
    for _ in range(int(20 / dt)):
        state_e = step(state_e, p_euler, harvest=0.0)
        state_r = step(state_r, p_rk45, harvest=0.0)
        fish_e.append(state_e.fish)
        fish_r.append(state_r.fish)
    t = np.arange(len(fish_e)) * dt
    ax.plot(t, fish_e, label="Euler", lw=1.5)
    ax.plot(t, fish_r, label="RK45", lw=1.5)
    ax.set_title(f"fish biomass, dt = {dt}")
    ax.set_xlabel("time")
    ax.legend()
axes[0].set_ylabel("fish")
plt.tight_layout()
plt.show()

## 4. À retenir

- Le modèle Lotka-Volterra **oscille** naturellement (pas de pression de pêche).
- Une **pression constante modérée** déplace l'équilibre vers le bas.
  Une pression trop forte fait disparaître le stock.
- **Euler diverge** quand `dt` croît ; **RK45 reste fidèle** car il subdivise
  le pas en interne.
- C'est ce module qui sera **wrappé en environnement Gymnasium** dans la
  **Brique 2**.